# Vietnamese OCR Toolbox - Inference local

Notebook này dựa trên `[vnm_ocr_toolbox]_Inference.ipynb`, nhưng dùng trực tiếp code trong folder gốc của repo hiện tại. Không có bước `git clone`, `git checkout`, `git pull` hay `%cd /content/main`.

Bạn chỉ cần chỉnh `IMAGE_PATH` và các đường dẫn weight trong phần cấu hình bên dưới nếu muốn chạy với ảnh/model khác.

In [1]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'run.py').exists() and (path / 'modules').exists() and (path / 'tool').exists():
            return path
    raise RuntimeError('Không tìm thấy repo root. Hãy mở notebook trong folder vietnamese-ocr-toolbox-master.')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Python:', sys.executable)

Repo root: /Users/nguyenbuihoanglong/Desktop/CBT/OCR/vietnamese-ocr-toolbox-master
Python: /opt/homebrew/Caskroom/miniconda/base/envs/mynotebook/bin/python


## Cài dependencies nếu môi trường chưa có

Cell dưới đây dùng requirements nhẹ cho inference trên Colab. Không nên cài thẳng `requirements.txt` gốc trên Colab mới vì file đó pin `torchvision==0.10` và có `pandas_profiling`, đều dễ lỗi với Python/Torch hiện tại. Nếu bạn đã cài dependencies rồi thì bỏ qua.

In [ ]:
# %pip install -r requirements-colab-inference.txt

In [ ]:
import yaml
import torch

# Repo gốc có hàm tải config OCR từ raw.githubusercontent.com của VietOCR.
# Patch lại để đọc các YAML đã có sẵn trong folder gốc: tool/config/ocr/models.
from modules.ocr.tool import config as ocr_tool_config

LOCAL_OCR_MODEL_CONFIG_DIR = REPO_ROOT / 'tool' / 'config' / 'ocr' / 'models'

def load_local_ocr_config(config_file_name):
    config_path = LOCAL_OCR_MODEL_CONFIG_DIR / config_file_name
    if not config_path.exists():
        raise FileNotFoundError(f'Không tìm thấy OCR config local: {config_path}')
    with open(config_path, encoding='utf-8') as f:
        return yaml.safe_load(f)

ocr_tool_config.download_config = load_local_ocr_config

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('OCR config source:', LOCAL_OCR_MODEL_CONFIG_DIR)
print('Torch CUDA available:', torch.cuda.is_available())
print('OCR device:', DEVICE)

if not torch.cuda.is_available():
    print('Lưu ý: module Detection/PAN của repo này đang hard-code torch.device("cuda").')
    print('Bạn cần môi trường có CUDA để chạy detection nguyên bản, hoặc sửa module detection để hỗ trợ CPU.')

## Chuẩn bị weights

Notebook ưu tiên dùng file trong `./weights`. Nếu file thiếu và `DOWNLOAD_MISSING_WEIGHTS = True`, nó sẽ tải pretrained weights bằng helper `tool.utils.download_pretrained_weights` có sẵn trong repo. Bước này không dùng git.

In [ ]:
from tool.utils import download_pretrained_weights

WEIGHTS_DIR = REPO_ROOT / 'weights'
WEIGHTS_DIR.mkdir(exist_ok=True)

DOWNLOAD_MISSING_WEIGHTS = True

DET_WEIGHT = WEIGHTS_DIR / 'PANNet_best_map.pth'
OCR_WEIGHT = WEIGHTS_DIR / 'transformerocr.pth'
BERT_WEIGHT = WEIGHTS_DIR / 'phobert_report.pth'

weight_jobs = [
    ('pan_resnet18_sroie19', DET_WEIGHT),
    ('transformerocr_mcocr', OCR_WEIGHT),
    ('phobert_mcocr', BERT_WEIGHT),
]

for model_name, weight_path in weight_jobs:
    if weight_path.exists():
        print('OK:', weight_path)
    elif DOWNLOAD_MISSING_WEIGHTS:
        print('Downloading:', model_name, '->', weight_path)
        download_pretrained_weights(model_name, cached=str(weight_path))
    else:
        print('Missing:', weight_path)

## Cấu hình input/output

In [ ]:
IMAGE_PATH = REPO_ROOT / 'demo' / 'data samples' / 'sroie19_1.png'
OUTPUT_DIR = REPO_ROOT / 'results' / IMAGE_PATH.stem

DO_RETRIEVE = True
FIND_BEST_ROTATION = False

CLASS_MAPPING = {'SELLER': 0, 'ADDRESS': 1, 'TIMESTAMP': 2, 'TOTAL_COST': 3, 'NONE': 4}
IDX_MAPPING = {v: k for k, v in CLASS_MAPPING.items()}

print('Image:', IMAGE_PATH)
print('Output:', OUTPUT_DIR)

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_bgr = cv2.imread(str(IMAGE_PATH))
if img_bgr is None:
    raise FileNotFoundError(f'Không đọc được ảnh: {IMAGE_PATH}')

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.axis('off');

## Khởi tạo pipeline bằng code local

In [ ]:
import shutil
import numpy as np
from PIL import Image

import modules.ocr as ocr
from modules import Preprocess, Detection, Retrieval, Correction
from modules.ocr.tool.predictor import Predictor
from tool.config import Config
from tool.utils import natural_keys, visualize, find_highest_score_each_class

class LocalOCR:
    def __init__(self, config_path='tool/config/ocr/configs.yaml', weight_path=None, device=DEVICE):
        main_config = Config(config_path)
        ocr_config = ocr.Config.load_config_from_name(main_config.model_name)
        ocr_config['cnn']['pretrained'] = False
        ocr_config['device'] = device
        ocr_config['predictor']['beamsearch'] = False
        ocr_config['weights'] = str(weight_path or OCR_WEIGHT)
        self.model = Predictor(ocr_config)

    def __call__(self, img, return_prob=False):
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img)
        return self.model.predict(img, return_prob)

    def predict_folder(self, img_paths, return_probs=False):
        texts = []
        probs = []
        for img_path in img_paths:
            img = Image.open(img_path)
            if return_probs:
                text, prob = self(img, True)
                texts.append(text)
                probs.append(prob)
            else:
                texts.append(self(img, False))
        return (texts, probs) if return_probs else texts

det_model = Detection(weight_path=str(DET_WEIGHT))
ocr_model = LocalOCR(weight_path=str(OCR_WEIGHT))
preproc = Preprocess(det_model=det_model, ocr_model=ocr_model, find_best_rotation=FIND_BEST_ROTATION)
correction = Correction()
retrieval = Retrieval(CLASS_MAPPING, mode='all', bert_weight=str(BERT_WEIGHT)) if DO_RETRIEVE else None

print('Modules loaded from:', REPO_ROOT / 'modules')

## Chạy inference

In [ ]:
import os
import time

start = time.time()

cache_dir = OUTPUT_DIR / 'cache'
crop_dir = cache_dir / 'crops'
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
cache_dir.mkdir(parents=True, exist_ok=True)

# Preprocess: input là BGR giống notebook gốc; DocScanner trả về ảnh RGB.
img_preprocessed = preproc(img_bgr)
cv2.imwrite(str(cache_dir / 'preprocessed.jpg'), cv2.cvtColor(img_preprocessed, cv2.COLOR_RGB2BGR))

boxes, img_detected = det_model(
    img_preprocessed,
    crop_region=True,
    return_result=True,
    output_path=str(cache_dir),
)
cv2.imwrite(str(cache_dir / 'detected.jpg'), cv2.cvtColor(img_detected, cv2.COLOR_RGB2BGR))

img_paths = os.listdir(crop_dir)
img_paths.sort(key=natural_keys)
img_paths = [str(crop_dir / item) for item in img_paths]

texts = ocr_model.predict_folder(img_paths, return_probs=False)
texts = correction(texts, return_score=False)

if DO_RETRIEVE:
    preds, probs = retrieval(texts)
else:
    preds, probs = None, None

result_img = OUTPUT_DIR / 'result.jpg'
visualize(
    img_preprocessed,
    boxes,
    texts,
    img_name=str(result_img),
    class_mapping=CLASS_MAPPING,
    labels=preds,
    probs=probs,
    visualize_best=DO_RETRIEVE,
)

if DO_RETRIEVE:
    best_score_idx = find_highest_score_each_class(preds, probs, CLASS_MAPPING)
    with open(OUTPUT_DIR / 'result.txt', 'w', encoding='utf-8') as f:
        for cls, idx in enumerate(best_score_idx):
            value = texts[idx] if idx >= 0 else ''
            f.write(f'{IDX_MAPPING[cls]} : {value}\n')

print(f'Executed in {time.time() - start:.2f}s')
print('Detected crops:', len(img_paths))
print('Result image:', result_img)
if DO_RETRIEVE:
    print('Result text:', OUTPUT_DIR / 'result.txt')

In [ ]:
from IPython.display import Image as IPyImage, display

display(
    IPyImage(filename=str(IMAGE_PATH), width=600),
    IPyImage(filename=str(cache_dir / 'preprocessed.jpg'), width=600),
    IPyImage(filename=str(cache_dir / 'detected.jpg'), width=600),
    IPyImage(filename=str(OUTPUT_DIR / 'result.jpg'), width=600),
)

if DO_RETRIEVE:
    print((OUTPUT_DIR / 'result.txt').read_text(encoding='utf-8'))
else:
    for text in texts:
        print(text)